In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

def save_sweep_hdf5(filename, traces, metadata_list, adc_rate, unit=None):
    """
    traces: list of 1D numpy arrays
    metadata_list: list of dicts, one per trace
    adc_rate: sampling rate in Hz, stored once at file level
    unit: optional time unit string ('s', 'ms', 'us', 'ns'); if None, computed automatically
    """
    if unit is None:
        n = len(traces[0])
        duration_s = n / adc_rate
        unit, _ = auto_unit(duration_s)

    with h5py.File(filename, 'w') as f:
        f.attrs['adc_rate'] = adc_rate
        f.attrs['unit'] = unit
        for i, (trace, meta) in enumerate(zip(traces, metadata_list)):
            dset = f.create_dataset(f'trace_{i}', data=trace, compression='gzip', compression_opts=4)
            for key, val in meta.items():
                dset.attrs[key] = val

def load_sweep_hdf5(filename):
    traces = []
    metadata_list = []
    with h5py.File(filename, 'r') as f:
        adc_rate = f.attrs['adc_rate']
        unit = f.attrs.get('unit', None)  # .get() in case an old file predates this attribute
        for key in f.keys():
            dset = f[key]
            traces.append(dset[:])
            metadata_list.append(dict(dset.attrs))
    return traces, metadata_list, adc_rate, unit

def load_matching(filename, predicate):
    matches = []
    metas = []
    with h5py.File(filename, 'r') as f:
        adc_rate = f.attrs['adc_rate']
        unit = f.attrs.get('unit', None)
        for key in f.keys():
            dset = f[key]
            attrs = dict(dset.attrs)
            if predicate(attrs):
                matches.append(dset[:])
                metas.append(attrs)
    return matches, metas, adc_rate, unit

def auto_unit(duration_s):
    if duration_s >= 1:
        return 's', 1
    elif duration_s >= 1e-3:
        return 'ms', 1e3
    elif duration_s >= 1e-6:
        return 'us', 1e6
    else:
        return 'ns', 1e9

def plot_multiple_traces(traces, labels=None, title="Power traces", unit=None, adc_rate=None, cmap_name='viridis'):
    if adc_rate is None:
        raise ValueError("adc_rate must be provided (no live 'scope' available when loading from file)")
    if len(traces) == 0:
        print("No traces to plot")
        return

    n = len(traces[0])
    duration_s = n / adc_rate

    if unit is None:
        unit, scale = auto_unit(duration_s)
    else:
        scale = {'s': 1, 'ms': 1e3, 'us': 1e6, 'ns': 1e9}[unit]

    t = (np.arange(n) / adc_rate) * scale
    #cmap = cm.get_cmap(cmap_name, len(traces))
    cmap = plt.colormaps[cmap_name].resampled(len(traces))

    plt.figure(figsize=(12, 5))
    for i, trace in enumerate(traces):
        label = labels[i] if labels is not None and i < len(labels) else f"trace {i}"
        plt.plot(t, trace, color=cmap(i), linewidth=0.8, alpha=0.8, label=label)

    plt.xlabel(f"time ({unit})")
    plt.ylabel("ADC value")
    plt.title(title)
    plt.legend(loc='upper right', fontsize=8, ncol=2)
    plt.tight_layout()
    plt.show()

import matplotlib.pyplot as plt
import numpy as np

def plot_traces_by_color_field(traces, metas, title="Power traces", unit=None, adc_rate=None, default_color='gray', ax=None, show=True):
    if adc_rate is None:
        raise ValueError("adc_rate must be provided")
    if len(traces) == 0:
        print("No traces to plot")
        return ax

    n = len(traces[0])
    duration_s = n / adc_rate

    if unit is None:
        unit, scale = auto_unit(duration_s)
    else:
        scale = {'s': 1, 'ms': 1e3, 'us': 1e6, 'ns': 1e9}[unit]

    t = (np.arange(n) / adc_rate) * scale

    if ax is None:
        fig, ax = plt.subplots(figsize=(12, 5))

    # collect labels already present on this axes so repeated calls don't duplicate legend entries
    existing_labels = {line.get_label() for line in ax.get_lines()}

    for trace, meta in zip(traces, metas):
        color = meta.get('color', default_color)
        label = meta.get('label', color)
        plot_label = label if label not in existing_labels else None
        existing_labels.add(label)
        ax.plot(t, trace, color=color, linewidth=0.8, alpha=0.8, label=plot_label)

    ax.set_xlabel(f"time ({unit})")
    ax.set_ylabel("ADC value")
    ax.set_title(title)
    ax.legend(loc='upper right', fontsize=8)

    if show:
        plt.tight_layout()
        plt.show()

    return ax


def auto_unit(duration_s):
    """Pick a sensible time unit based on total duration in seconds."""
    if duration_s >= 1:
        return 's', 1
    elif duration_s >= 1e-3:
        return 'ms', 1e3
    elif duration_s >= 1e-6:
        return 'us', 1e6
    else:
        return 'ns', 1e9

def calc_unit(one_trace, adc_rate):
    n = len(one_trace)
    duration_s = n / adc_rate
    unit, scale = auto_unit(duration_s)
    #print(f"n={n}, adc_rate={adc_rate}, duration={duration_s*1e6:.1f} us, chosen unit={unit}")
    return unit, scale
        